####Transform Orders Data
- Access elements from JSON object
- Deduplicate Array Elements
- Explode Arrays
- Write transformed data to silver layer

#### 1. Access elements from JSON object

In [0]:
%sql
SELECT
    json_string.customer_id,
    json_string.order_date,
    json_string.order_id,
    json_string.order_status,
    json_string.payment_method,
    json_string.total_amount,
    json_string.transaction_timestamp,
    json_string.items,
    json_string
FROM gizmobox.silver.orders_json

####2. Deduplicate Array Elements
- https://docs.databricks.com/aws/en/sql/language-manual/functions/array_distinct

In [0]:
%sql
SELECT
    json_string.customer_id,
    json_string.order_date,
    json_string.order_id,
    json_string.order_status,
    json_string.payment_method,
    json_string.total_amount,
    json_string.transaction_timestamp,
    array_distinct(json_string.items),
    json_string
FROM gizmobox.silver.orders_json

####3. Explode Arrays
- https://docs.databricks.com/aws/en/pyspark/reference/functions/explode

In [0]:
%sql
SELECT
    json_string.customer_id,
    json_string.order_date,
    json_string.order_id,
    json_string.order_status,
    json_string.payment_method,
    json_string.total_amount,
    json_string.transaction_timestamp,
    explode(array_distinct(json_string.items)) AS items,
    json_string
FROM gizmobox.silver.orders_json

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW tv_orders_exploded AS
SELECT
    json_string.customer_id,
    json_string.order_date,
    json_string.order_id,
    json_string.order_status,
    json_string.payment_method,
    json_string.total_amount,
    json_string.transaction_timestamp,
    explode(array_distinct(json_string.items)) AS items,
    json_string
FROM gizmobox.silver.orders_json

In [0]:
%sql
SELECT * FROM tv_orders_exploded

In [0]:
%sql

SELECT
    customer_id,
    order_date,
    order_id,
    order_status,
    payment_method,
    total_amount,
    transaction_timestamp,
    items.item_id,
    items.name,
    items.price,
    items.quantity,
    items.category,
    items.details.brand,
    items.details.color
FROM tv_orders_exploded

####4. Write transformed data to silver layer 

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gizmobox.silver.orders
AS
SELECT
    customer_id,
    order_date,
    order_id,
    order_status,
    payment_method,
    total_amount,
    transaction_timestamp,
    items.item_id,
    items.name,
    items.price,
    items.quantity,
    items.category,
    items.details.brand,
    items.details.color
FROM tv_orders_exploded

In [0]:
%sql
SELECT * FROM gizmobox.silver.orders